# 02 · 日汇总构建（M2）

演示 `demandlens.data.aggregate` 的聚合流程并在小文件上真实执行，然后加载正式产物 `daily_all_categories.csv`（多类别商店 7 个月，由 `scripts/build_daily.py` 在本机批量跑出）做汇总核对。

## 流程（对应作业 3.5）

原始事件 CSV → 分块读取（100 万行/块）→ 解析时间与事件类型 → 按日期 × 类别聚合 → 每日 Views / Cart / Purchase → 核对统计。一致性不变式由 `tests/test_aggregate.py` 覆盖。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (cand / 'src' / 'demandlens').exists():
        ROOT = cand
        break
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import kagglehub
from demandlens.data.aggregate import aggregate_month

print('ready')

ready


## 在演示文件上执行聚合

In [2]:
RAW = kagglehub.dataset_download(
    'mkechinov/ecommerce-events-history-in-cosmetics-shop',
    path='2019-Oct.csv')
daily, stats = aggregate_month(RAW, chunksize=500_000, progress_every=4)
print('原始行数:', f"{stats['raw_rows']:,}")
print('进入聚合行数:', f"{stats['aggregatable_rows']:,}")
print('契约事件计数:', dict(stats['contract_kept']))
daily.head()

    2019-Oct.csv: 已处理 4 块 / 2,000,000 行


    2019-Oct.csv: 已处理 8 块 / 4,000,000 行


原始行数: 4,102,283
进入聚合行数: 3,340,173
契约事件计数: {'view': 1862164, 'cart': 1232385, 'purchase': 245624}


event_type,date,category_id,category_code,views,cart,purchase
0,2019-10-01,1487580004832248652,None,364.0,510.0,31.0
1,2019-10-01,1487580004857414477,None,174.0,183.0,4.0
2,2019-10-01,1487580004882580302,None,156.0,109.0,24.0
3,2019-10-01,1487580004916134735,None,454.0,730.0,86.0
4,2019-10-01,1487580004983243602,None,7.0,3.0,0.0


## 聚合一致性核对（D2）

进入聚合的各事件类型计数必须等于日汇总列之和。

In [3]:
checks = {
    'view': (stats['contract_kept']['view'], int(daily['views'].sum())),
    'cart': (stats['contract_kept']['cart'], int(daily['cart'].sum())),
    'purchase': (stats['contract_kept']['purchase'], int(daily['purchase'].sum())),
}
pd.DataFrame([(k, a, b, b - a) for k, (a, b) in checks.items()],
             columns=['event', 'raw_kept', 'aggregated', 'diff'])

,event,raw_kept,aggregated,diff
0,view,1862164,1862164,0
1,cart,1232385,1232385,0
2,purchase,245624,245624,0


## 正式产物：多类别商店日汇总

下面加载 `data/processed/daily_all_categories.csv`（若尚未生成，先运行 `scripts/run_m2_pipeline.sh`）。

In [4]:
from demandlens.data.loader import CONTRACT_EVENTS

OUT = ROOT / 'data' / 'processed' / 'daily_all_categories.csv'
if OUT.exists():
    full = pd.read_csv(OUT, parse_dates=['date'])
    print('行数:', f'{len(full):,}')
    print('日期:', full['date'].min().date(), '..', full['date'].max().date())
    print('类别数:', full['category_id'].nunique())
    print('日购买事件 总量/日均:', int(full['purchase'].sum()),
          round(full['purchase'].mean(), 1))
    print('每日合计的购买事件 分布:')
    per_day = full.groupby('date')['purchase'].sum()
    print(per_day.describe().round(1))
else:
    print('正式产物尚未生成（多月份聚合仍在进行或未运行）')

行数: 36,555
日期: 2019-10-01 .. 2019-11-30
类别数: 691
日购买事件 总量/日均: 1659788 45.4
每日合计的购买事件 分布:
count        61.0
mean      27209.6
std       21726.4
min           0.0
25%       22458.0
50%       24305.0
75%       25385.0
max      185195.0
Name: purchase, dtype: float64


## 交接说明

- 本表**不补零、不筛选类别**（契约 3.6 的 M2 产物）。
- M3 依据质量审核与类别选择产出 `daily_category.csv` 后，下游只用后者。
- 核对明细：`reports/tables/aggregation_check.csv`；报告解读：`reports/aggregation_check_report.md`。